# Targeted H1 imbalance validation — Gaussian, high-$B_{ref}$

Self-contained validation notebook for testing whether imbalance worsens the unequal-sample EC approximation under $H_1$ when the MC reference is better resolved.

Primary outputs: unresolved-reference heatmap, formula MSE vs imbalance, paired fraction-worse vs imbalance with $R_{valid}$ annotations, relative $B_{eq}$ heatmaps, and equivalent-MC-runtime curves.


## Cluster execution notes

This version is **OpenPBS-aware** and remains usable locally.

When executed inside a PBS job:

- `PBS_O_WORKDIR` is used as the project/submission directory.
- Persistent results are saved to `/work/<user>/thesis_results/<run_name>/`.
- Active checkpoint I/O is performed in job-specific `/scratch_local/...`.
- The checkpoint is mirrored to `/work` so future jobs can resume safely.
- Matplotlib uses the non-interactive `Agg` backend.
- All figures are saved automatically.
- Numerical outputs and metadata are saved automatically.
- A configuration hash is built into the run folder name, preventing stale checkpoint reuse when \(B_{\rm ref}\), \(R\), \(N\), or the ratio grid change.


In [ ]:

# ============================================================
# IMPORTS + EXECUTION ENVIRONMENT
# ============================================================

import os
import re
import json
import math
import socket
import hashlib
import shutil
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

IS_PBS = bool(os.environ.get("PBS_JOBID"))

import matplotlib
if IS_PBS:
    matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import gammaln, logsumexp
from scipy.stats import norm
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
np.set_printoptions(precision=6, suppress=True)

RUN_START = perf_counter()


In [ ]:
# 2. Data generation + Monte Carlo

def deterministic_seed(*parts):
    ss=np.random.SeedSequence([int(p) for p in parts])
    return int(ss.generate_state(1,dtype=np.uint32)[0])

def ratio_to_sizes(N,r):
    n1=int(round(r*N)); n2=N-n1
    if min(n1,n2)<2: raise ValueError((N,r,n1,n2))
    return n1,n2

def generate_paired_H1(N,ratios,seed):
    rng=np.random.default_rng(seed)
    max_n1=max(ratio_to_sizes(N,r)[0] for r in ratios)
    max_n2=max(ratio_to_sizes(N,r)[1] for r in ratios)
    xfull=rng.normal(MEAN_X,SD,max_n1)
    yfull=rng.normal(MEAN_Y,SD,max_n2)
    return {float(r):(xfull[:ratio_to_sizes(N,r)[0]].copy(),yfull[:ratio_to_sizes(N,r)[1]].copy()) for r in ratios}

def diffmeans(x,y): return float(np.mean(x)-np.mean(y))

def extreme(stats,obs): return np.abs(stats)>=abs(obs)

def resolution(K):
    if K==0: return 'unresolved'
    if K<MIN_EXTREME_COUNT: return 'low_resolution'
    return 'well_resolved'

def mc_test(x,y,B,rng,batch_size):
    x=np.asarray(x,float); y=np.asarray(y,float)
    n1,n2=len(x),len(y); N=n1+n2
    pool=np.concatenate([x,y]); total=float(pool.sum()); obs=diffmeans(x,y)
    K=0; done=0; elapsed=0.0
    while done<B:
        b=min(batch_size,B-done); t0=perf_counter()
        keys=rng.random((b,N))
        idx=np.argpartition(keys,kth=n1-1,axis=1)[:,:n1]
        s1=pool[idx].sum(axis=1); s2=total-s1
        stats=s1/n1-s2/n2
        K += int(extreme(stats,obs).sum())
        elapsed += perf_counter()-t0; done += b
    if K>0:
        logp=float(np.log(K)-np.log(B)); p=K/B; S=-logp/np.log(10.0)
    else:
        logp=float('-inf'); p=0.0; S=float('nan')
    return dict(p=p,logp=logp,surprisal=S,K=int(K),status=resolution(K),obs=obs,time_per_perm=elapsed/B)


In [ ]:
# 3. Unequal EC approximation in log space

def ec_log_weights(n1,n2):
    k=np.arange(max(0,n1-n2),n1+1,dtype=int); l=n1-k
    a=gammaln(n1+1)-gammaln(k+1)-gammaln(n1-k+1)
    b=gammaln(n2+1)-gammaln(l+1)-gammaln(n2-l+1)
    lw=a+b; lw-=logsumexp(lw)
    return k,lw

def class_logtail(obs,mu,var):
    var=float(max(var,0.0))
    if var==0.0: return 0.0 if abs(mu)>=abs(obs) else float('-inf')
    sd=np.sqrt(var); t=abs(obs)
    return min(float(np.logaddexp(norm.logcdf((-t-mu)/sd),norm.logsf((t-mu)/sd))),0.0)

def ec_approx(x,y,a,b,method):
    n1,n2=len(x),len(y); N=n1+n2; obs=diffmeans(x,y)
    k,lw=ec_log_weights(n1,n2); l=n1-k
    ma,mb=float(np.mean(a)),float(np.mean(b))
    va,vb=float(np.var(a,ddof=1)),float(np.var(b,ddof=1))
    mus=((N*k-n1**2)/(n1*n2))*(ma-mb)
    scale=(N/(n1*n2))**2
    vars_=scale*((k*(n1-k)/n1)*va + (l*(n2-l)/n2)*vb)
    lt=np.array([class_logtail(obs,m,v) for m,v in zip(mus,vars_)])
    logp=min(float(logsumexp(lw+lt)),0.0)
    return dict(method=method,logp=logp,p=float(np.exp(logp)),surprisal=-logp/np.log(10.0))

def ordered_result(x,y):
    n1=len(x); pool=np.sort(np.concatenate([x,y])); a=pool[-n1:].copy(); b=pool[:-n1].copy()
    return ec_approx(x,y,a,b,'ordered')

def original_result(x,y): return ec_approx(x,y,np.asarray(x).copy(),np.asarray(y).copy(),'original')
ESTIMATORS={'ordered':ordered_result,'original':original_result}

def log_abs_exp_diff(a,b):
    if np.isneginf(a) and np.isneginf(b): return float('-inf')
    if a==b: return float('-inf')
    hi,lo=max(a,b),min(a,b)
    if np.isneginf(lo): return hi
    return float(hi+np.log(-np.expm1(lo-hi)))

def log_p1mp(logp):
    if np.isneginf(logp) or logp>=0: return float('-inf')
    return float(logp+np.log(-np.expm1(logp)))

def safe_exp(z):
    if np.isnan(z): return float('nan')
    if np.isposinf(z): return float('inf')
    if np.isneginf(z): return 0.0
    return float(np.exp(z)) if z<np.log(np.finfo(float).max) else float('inf')


In [ ]:
# 4. Runner with checkpointing and ETA progress bar

def run_validation(resume=True):
    if resume and CHECKPOINT.exists():
        raw=pd.read_csv(CHECKPOINT); print('Resuming from',len(raw),'rows')
    else: raw=pd.DataFrame()
    if raw.empty: done=set()
    else:
        counts=raw.groupby(['N','run','ratio'])['method'].nunique()
        done={tuple(i) for i,n in counts.items() if n==len(ESTIMATORS)}
    total=len(N_VALUES)*len(RATIOS)*R
    bar=tqdm(total=total,desc='High-Bref H1 validation',unit='scenario',dynamic_ncols=True)
    bar.update(len(done)); new=[]
    try:
        for Ni,N in enumerate(N_VALUES):
            for run in range(R):
                dseed=deterministic_seed(MASTER_SEED,1000,Ni,run)
                datasets=generate_paired_H1(N,RATIOS,dseed)
                added=False
                for ri,r in enumerate(RATIOS):
                    key=(int(N),int(run),float(r))
                    if key in done: continue
                    x,y=datasets[float(r)]; n1,n2=len(x),len(y)
                    mseed=deterministic_seed(MASTER_SEED,2000,Ni,run,ri)
                    mc=mc_test(x,y,B_REF,np.random.default_rng(mseed),BATCH_SIZE)
                    for method,est in ESTIMATORS.items():
                        t0=perf_counter(); fo=est(x,y); ftime=perf_counter()-t0
                        if mc['K']==0:
                            lognum=float('-inf'); logsq=float('nan')
                        else:
                            lognum=log_p1mp(mc['logp']); logsq=2*log_abs_exp_diff(fo['logp'],mc['logp'])
                        new.append(dict(N=N,run=run,ratio=r,n1=n1,n2=n2,method=method,data_seed=dseed,mc_seed=mseed,B_ref=B_REF,
                                        p_ref=mc['p'],log_p_ref=mc['logp'],surprisal_ref=mc['surprisal'],mc_extreme_count=mc['K'],mc_resolution_status=mc['status'],mc_time_per_permutation=mc['time_per_perm'],
                                        p_formula=fo['p'],log_p_formula=fo['logp'],surprisal_formula=fo['surprisal'],formula_time_seconds=ftime,
                                        log_mc_numerator=lognum,log_formula_squared_error=logsq))
                    done.add(key); added=True; bar.update(1)
                if added and new:
                    raw=pd.concat([raw,pd.DataFrame(new)],ignore_index=True); raw.to_csv(CHECKPOINT,index=False); new=[]
    finally: bar.close()
    return raw.sort_values(['N','run','ratio','method']).reset_index(drop=True)


In [ ]:
# 5. Summaries

def summarize(raw):
    rows=[]
    for (N,r,n1,n2,m),g in raw.groupby(['N','ratio','n1','n2','method'],sort=True):
        unresolved=int((g.mc_extreme_count==0).sum()); low=int((g.mc_resolution_status=='low_resolution').sum()); well=int((g.mc_resolution_status=='well_resolved').sum())
        logs=g.log_formula_squared_error.to_numpy(float); valid=np.isfinite(logs)
        lmse=float(logsumexp(logs[valid])-np.log(valid.sum())) if valid.any() else float('nan'); mse=safe_exp(lmse)
        if unresolved==0:
            lnum=float(logsumexp(g.log_mc_numerator.to_numpy(float))); lden=float(logsumexp(g.log_formula_squared_error.to_numpy(float)))
            lB=float('inf') if np.isneginf(lden) else lnum-lden; B=safe_exp(lB)
            c=float(np.median(g.mc_time_per_permutation)); teq=safe_exp(lB+np.log(c)) if np.isfinite(lB) and c>0 else float('nan')
        else: lB=B=teq=float('nan')
        rows.append(dict(N=N,ratio=r,n1=n1,n2=n2,method=m,R=len(g),R_valid_error=int(valid.sum()),well_resolved=well,low_resolution=low,unresolved=unresolved,
                         mean_formula_mse=mse,log_mean_formula_mse=lmse,log_aggregated_B_eq=lB,aggregated_B_eq=B,equivalent_mc_time_seconds=teq,
                         median_formula_time_seconds=float(np.median(g.formula_time_seconds))))
    return pd.DataFrame(rows)

def fraction_worse(raw):
    w=raw.copy(); w['err']=np.nan; v=np.isfinite(w.log_formula_squared_error); w.loc[v,'err']=np.exp(w.loc[v,'log_formula_squared_error'])
    rows=[]
    for (N,m,run),g in w.groupby(['N','method','run']):
        b=g.loc[np.isclose(g.ratio,0.5)]
        if b.empty or not np.isfinite(b.err.iloc[0]): continue
        base=b.err.iloc[0]
        for _,row in g.iterrows():
            if np.isfinite(row.err): rows.append(dict(N=N,method=m,run=run,ratio=row.ratio,imbalance_worse=row.err>base))
    if not rows: return pd.DataFrame()
    return pd.DataFrame(rows).groupby(['N','method','ratio'],as_index=False).agg(R_valid=('imbalance_worse','size'),fraction_worse=('imbalance_worse','mean'))


In [ ]:
# 6. Plots
METHODS=['ordered','original']

def plot_resolution(s):
    p=s.groupby(['N','ratio'],as_index=False).unresolved.max().pivot(index='ratio',columns='N',values='unresolved')
    fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(p.to_numpy(),aspect='auto',origin='lower')
    ax.set_xticks(range(len(p.columns))); ax.set_xticklabels([str(int(x)) for x in p.columns]); ax.set_yticks(range(len(p.index))); ax.set_yticklabels([f'{r:.2f}' for r in p.index])
    for i in range(p.shape[0]):
        for j in range(p.shape[1]): ax.text(j,i,f'{int(p.iloc[i,j])}/{R}',ha='center',va='center')
    ax.set(xlabel='Total sample size N',ylabel=r'Group-1 proportion $r=n_1/N$',title='Gaussian H1 — unresolved MC references (K=0)')
    plt.colorbar(im,ax=ax,label='Number unresolved'); plt.tight_layout(); plt.show()

def plot_mse(s):
    fig,axs=plt.subplots(1,2,figsize=(15,5.5),sharey=True)
    for ax,m in zip(axs,METHODS):
        dm=s[s.method==m]
        for N,d in dm.groupby('N'):
            d=d.sort_values('ratio'); v=np.isfinite(d.mean_formula_mse)&(d.mean_formula_mse>0); ax.plot(d.loc[v,'ratio'],d.loc[v,'mean_formula_mse'],marker='o',label=f'N={N}')
        ax.set_yscale('log'); ax.set_xlabel(r'Group-1 proportion $r=n_1/N$'); ax.set_title(m.capitalize()); ax.grid(True,which='both',alpha=.25)
    axs[0].set_ylabel('Average formula MSE'); h,l=axs[-1].get_legend_handles_labels(); fig.legend(h,l,loc='upper center',ncol=len(N_VALUES)); fig.suptitle('Gaussian H1 — high-$B_{ref}$ validation\nFormula MSE vs imbalance',y=1.05); plt.tight_layout(); plt.show()

def plot_fraction(p):
    fig,axs=plt.subplots(1,2,figsize=(15,5.5),sharey=True)
    for ax,m in zip(axs,METHODS):
        dm=p[(p.method==m)&(~np.isclose(p.ratio,0.5))]
        for N,d in dm.groupby('N'):
            d=d.sort_values('ratio'); ax.plot(d.ratio,d.fraction_worse,marker='o',label=f'N={N}')
            for _,row in d.iterrows(): ax.annotate(str(int(row.R_valid)),(row.ratio,row.fraction_worse),xytext=(0,8),textcoords='offset points',ha='center',fontsize=8)
        ax.axhline(.5,ls='--'); ax.set_ylim(-.03,1.08); ax.set_xlabel(r'Group-1 proportion $r=n_1/N$'); ax.set_title(m.capitalize()); ax.grid(True,alpha=.25)
    axs[0].set_ylabel('Fraction of paired datasets where\nimbalance increases squared error'); h,l=axs[-1].get_legend_handles_labels(); fig.legend(h,l,loc='upper center',ncol=len(N_VALUES)); fig.suptitle('Gaussian H1 — does imbalance consistently worsen the approximation?\nNumbers above points = R_valid',y=1.08); plt.tight_layout(); plt.show()

def plot_relative_Beq(s):
    for m in METHODS:
        d=s[s.method==m].copy(); d['rel']=np.nan
        for N,g in d.groupby('N'):
            b=g[np.isclose(g.ratio,.5)]
            if b.empty or not np.isfinite(b.log_aggregated_B_eq.iloc[0]): continue
            base=b.log_aggregated_B_eq.iloc[0]
            idx=g.index; v=np.isfinite(d.loc[idx,'log_aggregated_B_eq']); d.loc[idx[v],'rel']=(d.loc[idx[v],'log_aggregated_B_eq']-base)/np.log(10)
        p=d.pivot(index='ratio',columns='N',values='rel'); fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(p.to_numpy(),aspect='auto',origin='lower')
        ax.set_xticks(range(len(p.columns))); ax.set_xticklabels([str(int(x)) for x in p.columns]); ax.set_yticks(range(len(p.index))); ax.set_yticklabels([f'{r:.2f}' for r in p.index])
        for i in range(p.shape[0]):
            for j in range(p.shape[1]):
                if np.isfinite(p.iloc[i,j]): ax.text(j,i,f'{p.iloc[i,j]:.2f}',ha='center',va='center')
        ax.set(xlabel='Total sample size N',ylabel=r'Group-1 proportion $r=n_1/N$',title=f'Gaussian H1 — {m.capitalize()}\n'+r'$\log_{10}[B_{eq}(N,r)/B_{eq}(N,0.5)]$')
        plt.colorbar(im,ax=ax,label='log10 relative B_eq'); plt.tight_layout(); plt.show()

def plot_time(s):
    ratios=np.array(RATIOS,float); cmap=plt.get_cmap('Blues'); sc=(ratios-ratios.min())/(ratios.max()-ratios.min()); colors={float(r):cmap(.3+.65*z) for r,z in zip(ratios,sc)}
    fig,axs=plt.subplots(1,2,figsize=(15,5.5))
    for ax,m in zip(axs,METHODS):
        dm=s[s.method==m]
        for r in RATIOS:
            d=dm[np.isclose(dm.ratio,r)].sort_values('N'); v=np.isfinite(d.equivalent_mc_time_seconds)&(d.equivalent_mc_time_seconds>0)
            if v.any(): ax.plot(d.loc[v,'N'],d.loc[v,'equivalent_mc_time_seconds'],marker='o',color=colors[float(r)],label=f'r={r:.2f}')
        f=dm.groupby('N',as_index=False).median(numeric_only=True); ax.plot(f.N,f.median_formula_time_seconds,marker='s',ls='--',color='black',label='Formula runtime')
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('Total sample size N'); ax.set_title(m.capitalize()); ax.grid(True,which='both',alpha=.25)
    axs[0].set_ylabel('Runtime (seconds)'); h,l=axs[-1].get_legend_handles_labels(); fig.legend(h,l,loc='upper center',ncol=6); fig.suptitle('Gaussian H1 — equivalent MC runtime by imbalance ratio',y=1.03); plt.tight_layout(); plt.show()


In [ ]:
# 7. Run experiment
raw_results=run_validation(resume=True)
summary_results=summarize(raw_results)
paired_results=fraction_worse(raw_results)
summary_results.to_csv(SUMMARY_FILE,index=False); paired_results.to_csv(PAIRED_FILE,index=False)
print('Finished:',len(raw_results),'raw rows')
display(summary_results[['N','ratio','method','R','R_valid_error','well_resolved','low_resolution','unresolved','mean_formula_mse','aggregated_B_eq','equivalent_mc_time_seconds']])
print('Paired results'); display(paired_results)


In [ ]:
# 8. Supervisor-ready plots
plot_resolution(summary_results)
plot_mse(summary_results)
plot_fraction(paired_results)
plot_relative_Beq(summary_results)
plot_time(summary_results)


In [ ]:
# 9. Sanity checks
assert ratio_to_sizes(200,.5)==(100,100)
for n1,n2 in [(100,100),(140,60),(190,10)]:
    _,lw=ec_log_weights(n1,n2); assert np.allclose(np.exp(lw).sum(),1.0)
p=.2; q=.18; direct=p*(1-p)/(q-p)**2; stable=np.exp(log_p1mp(np.log(p))-2*log_abs_exp_diff(np.log(q),np.log(p))); assert np.allclose(direct,stable,rtol=1e-12)
print('All sanity checks passed.')


# ============================================================
# FINALIZE RUN
# ============================================================

TOTAL_RUNTIME_SECONDS = perf_counter() - RUN_START

RUN_METADATA["finished_utc"] = datetime.now(timezone.utc).isoformat()
RUN_METADATA["total_runtime_seconds"] = float(TOTAL_RUNTIME_SECONDS)
RUN_METADATA["total_runtime_minutes"] = float(TOTAL_RUNTIME_SECONDS / 60.0)
RUN_METADATA["completed_successfully"] = True

with open(METADATA_FILE, "w") as f:
    json.dump(RUN_METADATA, f, indent=2)

print(f"\nTotal notebook runtime: {TOTAL_RUNTIME_SECONDS:.2f} s "
      f"({TOTAL_RUNTIME_SECONDS/60:.2f} min)")
print("Persistent outputs:", OUTPUT_DIR)
print("Metadata:", METADATA_FILE)

if IS_PBS and SCRATCH_DIR.exists():
    shutil.rmtree(SCRATCH_DIR, ignore_errors=True)
    print("Removed temporary scratch directory:", SCRATCH_DIR)
